# Multi-Disease Prediction System
## Full ML Pipeline: Data Cleaning → Feature Engineering → XGBoost Training → Evaluation

**Diseases Covered:**
- 🩺 Diabetes
- ❤️ Heart Disease
- 🫀 Liver Disease

In [ ]:
# ─── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc, classification_report
)

warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)
print('✅ Imports successful')

---
## 🩺 DISEASE 1: Diabetes

In [ ]:
# ─── Cell 2: Load Diabetes Data ────────────────────────────────────────────────
df_diabetes = pd.read_csv('../data/diabetes.csv')
print('Shape:', df_diabetes.shape)
print('\nFirst 5 rows:')
df_diabetes.head()

In [ ]:
# ─── Cell 3: Diabetes – Exploratory Data Analysis ──────────────────────────────
print('=== Missing Values ===')
print(df_diabetes.isnull().sum())

print('\n=== Data Types ===')
print(df_diabetes.dtypes)

print('\n=== Class Distribution ===')
print(df_diabetes['Outcome'].value_counts())

print('\n=== Basic Statistics ===')
df_diabetes.describe()

In [ ]:
# ─── Cell 4: Diabetes – Data Cleaning & Feature Engineering ────────────────────
df_d = df_diabetes.copy()

# STEP 1: Replace biologically impossible zeros with NaN
zero_not_allowed = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_not_allowed:
    zero_count = (df_d[col] == 0).sum()
    if zero_count > 0:
        print(f'[CLEAN] {col}: {zero_count} zeros → replacing with NaN')
        df_d[col] = df_d[col].replace(0, np.nan)

# STEP 2: Impute missing values with median (robust to outliers)
for col in zero_not_allowed:
    median_val = df_d[col].median()
    df_d[col] = df_d[col].fillna(median_val)
    print(f'[IMPUTE] {col}: filled NaN with median = {median_val:.2f}')

# STEP 3: Feature Engineering
# BMI category
df_d['BMI_Category'] = pd.cut(
    df_d['BMI'],
    bins=[0, 18.5, 25, 30, 100],
    labels=[0, 1, 2, 3]  # Underweight, Normal, Overweight, Obese
).astype(int)

# Age group
df_d['Age_Group'] = pd.cut(
    df_d['Age'],
    bins=[0, 30, 45, 60, 100],
    labels=[0, 1, 2, 3]
).astype(int)

# Glucose-Insulin ratio
df_d['Glucose_Insulin_Ratio'] = df_d['Glucose'] / (df_d['Insulin'] + 1)

print('\n✅ Diabetes data cleaned. Shape:', df_d.shape)

In [ ]:
# ─── Cell 5: Diabetes – Correlation Heatmap ────────────────────────────────────
plt.figure(figsize=(12, 8))
corr = df_d.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Diabetes Dataset – Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/diabetes_heatmap.png', dpi=150)
plt.show()
print('✅ Heatmap saved')

In [ ]:
# ─── Cell 6: Diabetes – Train XGBoost Model ────────────────────────────────────
X_d = df_d.drop('Outcome', axis=1)
y_d = df_d['Outcome']

# Scale features
scaler_d = StandardScaler()
X_d_scaled = scaler_d.fit_transform(X_d)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_d_scaled, y_d, test_size=0.2, random_state=42, stratify=y_d
)

# XGBoost model
model_diabetes = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model_diabetes.fit(X_train, y_train)

y_pred = model_diabetes.predict(X_test)
y_prob = model_diabetes.predict_proba(X_test)[:, 1]

print('=== DIABETES MODEL EVALUATION ===')
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ─── Cell 7: Diabetes – Confusion Matrix & Feature Importance ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
axes[0].set_title('Diabetes – Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature Importance
importance_df = pd.DataFrame({
    'Feature': X_d.columns,
    'Importance': model_diabetes.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(importance_df['Feature'], importance_df['Importance'],
             color=plt.cm.viridis(np.linspace(0.2, 0.9, len(importance_df))))
axes[1].set_title('Diabetes – Feature Importance (XGBoost)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../models/diabetes_evaluation.png', dpi=150)
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Diabetes – ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../models/diabetes_roc.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 8: Save Diabetes Model & Scaler ──────────────────────────────────────
joblib.dump(model_diabetes, '../models/diabetes_model.pkl')
joblib.dump(scaler_d, '../models/diabetes_scaler.pkl')
joblib.dump(list(X_d.columns), '../models/diabetes_features.pkl')
print('✅ Diabetes model, scaler, and feature list saved!')

---
## ❤️ DISEASE 2: Heart Disease

In [ ]:
# ─── Cell 9: Load & Clean Heart Disease Data ───────────────────────────────────
df_heart = pd.read_csv('../data/heart_disease.csv')
print('Shape:', df_heart.shape)
print('\nMissing values:\n', df_heart.isnull().sum())
print('\nSex distribution:\n', df_heart['sex'].value_counts())
print('\nCP types:\n', df_heart['cp'].value_counts())
df_heart.head()

In [ ]:
# ─── Cell 10: Heart Disease – Encoding & Feature Engineering ───────────────────
df_h = df_heart.copy()

# STEP 1: Encode categorical columns
le_sex = LabelEncoder()
df_h['sex'] = le_sex.fit_transform(df_h['sex'])  # male=1, female=0
print('[ENCODE] sex:', dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))

le_cp = LabelEncoder()
df_h['cp'] = le_cp.fit_transform(df_h['cp'])
print('[ENCODE] cp:', dict(zip(le_cp.classes_, le_cp.transform(le_cp.classes_))))

le_thal = LabelEncoder()
df_h['thal'] = le_thal.fit_transform(df_h['thal'].fillna('normal'))
print('[ENCODE] thal:', dict(zip(le_thal.classes_, le_thal.transform(le_thal.classes_))))

# STEP 2: Handle missing values
for col in df_h.columns:
    if df_h[col].isnull().sum() > 0:
        df_h[col] = df_h[col].fillna(df_h[col].median())
        print(f'[IMPUTE] {col}: filled with median')

# STEP 3: Feature engineering
df_h['age_thalach_ratio'] = df_h['age'] / (df_h['thalach'] + 1)  # Age to max HR ratio
df_h['bp_chol_index'] = df_h['trestbps'] * df_h['chol'] / 10000  # Combined risk index

print('\n✅ Heart disease data cleaned. Shape:', df_h.shape)

In [ ]:
# ─── Cell 11: Heart Disease – Correlation Heatmap ──────────────────────────────
plt.figure(figsize=(14, 9))
corr = df_h.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Heart Disease – Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/heart_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 12: Heart Disease – Train & Evaluate ─────────────────────────────────
X_h = df_h.drop('target', axis=1)
y_h = df_h['target']

scaler_h = StandardScaler()
X_h_scaled = scaler_h.fit_transform(X_h)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h_scaled, y_h, test_size=0.2, random_state=42, stratify=y_h
)

model_heart = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model_heart.fit(X_train_h, y_train_h)

y_pred_h = model_heart.predict(X_test_h)
y_prob_h = model_heart.predict_proba(X_test_h)[:, 1]

print('=== HEART DISEASE MODEL EVALUATION ===')
print(f'Accuracy  : {accuracy_score(y_test_h, y_pred_h):.4f}')
print(f'Precision : {precision_score(y_test_h, y_pred_h):.4f}')
print(f'Recall    : {recall_score(y_test_h, y_pred_h):.4f}')
print(f'F1 Score  : {f1_score(y_test_h, y_pred_h):.4f}')
print('\nClassification Report:')
print(classification_report(y_test_h, y_pred_h))

In [ ]:
# ─── Cell 13: Heart – Confusion Matrix & Feature Importance ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_h = confusion_matrix(y_test_h, y_pred_h)
sns.heatmap(cm_h, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
axes[0].set_title('Heart Disease – Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

importance_df_h = pd.DataFrame({
    'Feature': X_h.columns,
    'Importance': model_heart.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(importance_df_h['Feature'], importance_df_h['Importance'],
             color=plt.cm.plasma(np.linspace(0.2, 0.9, len(importance_df_h))))
axes[1].set_title('Heart Disease – Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../models/heart_evaluation.png', dpi=150)
plt.show()

# ROC
fpr_h, tpr_h, _ = roc_curve(y_test_h, y_prob_h)
roc_auc_h = auc(fpr_h, tpr_h)
plt.figure(figsize=(6,4))
plt.plot(fpr_h, tpr_h, color='crimson', lw=2, label=f'ROC AUC = {roc_auc_h:.2f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Heart Disease – ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../models/heart_roc.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 14: Save Heart Model ──────────────────────────────────────────────────
joblib.dump(model_heart, '../models/heart_model.pkl')
joblib.dump(scaler_h, '../models/heart_scaler.pkl')
joblib.dump(list(X_h.columns), '../models/heart_features.pkl')

# Save encoders for use during inference
joblib.dump({'sex': le_sex, 'cp': le_cp, 'thal': le_thal}, '../models/heart_encoders.pkl')
print('✅ Heart disease model, scaler, encoders saved!')

---
## 🫀 DISEASE 3: Liver Disease

In [ ]:
# ─── Cell 15: Load & Clean Liver Disease Data ──────────────────────────────────
df_liver = pd.read_csv('../data/liver_disease.csv')
print('Shape:', df_liver.shape)
print('\nMissing values:\n', df_liver.isnull().sum())
print('\nTarget distribution (1=Disease, 2=No Disease):\n', df_liver['Dataset'].value_counts())
df_liver.head()

In [ ]:
# ─── Cell 16: Liver Disease – Cleaning & Feature Engineering ───────────────────
df_l = df_liver.copy()

# STEP 1: Fix target column (1=disease, 2=no disease → convert to 0/1)
df_l['target'] = (df_l['Dataset'] == 1).astype(int)
df_l.drop('Dataset', axis=1, inplace=True)

# STEP 2: Encode Gender
le_gender = LabelEncoder()
df_l['Gender'] = le_gender.fit_transform(df_l['Gender'])  # Female=0, Male=1
print('[ENCODE] Gender:', dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))

# STEP 3: Handle missing values
missing_cols = df_l.isnull().sum()
missing_cols = missing_cols[missing_cols > 0]
if len(missing_cols) > 0:
    for col in missing_cols.index:
        df_l[col] = df_l[col].fillna(df_l[col].median())
        print(f'[IMPUTE] {col}: filled with median')
else:
    print('[INFO] No missing values found')

# STEP 4: Feature Engineering
df_l['AST_ALT_Ratio'] = df_l['Aspartate_Aminotransferase'] / (df_l['Alamine_Aminotransferase'] + 1)
df_l['Bilirubin_Ratio'] = df_l['Direct_Bilirubin'] / (df_l['Total_Bilirubin'] + 1)
df_l['Protein_Albumin_Diff'] = df_l['Total_Protiens'] - df_l['Albumin']

print('\n✅ Liver data cleaned. Shape:', df_l.shape)
print('Target distribution:\n', df_l['target'].value_counts())

In [ ]:
# ─── Cell 17: Liver – Correlation Heatmap ──────────────────────────────────────
plt.figure(figsize=(14, 9))
corr = df_l.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='YlOrRd', center=0, linewidths=0.5)
plt.title('Liver Disease – Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/liver_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 18: Liver Disease – Train & Evaluate ─────────────────────────────────
X_l = df_l.drop('target', axis=1)
y_l = df_l['target']

scaler_l = StandardScaler()
X_l_scaled = scaler_l.fit_transform(X_l)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_l_scaled, y_l, test_size=0.2, random_state=42, stratify=y_l
)

model_liver = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=(y_l == 0).sum() / (y_l == 1).sum(),  # Handle class imbalance
    random_state=42
)
model_liver.fit(X_train_l, y_train_l)

y_pred_l = model_liver.predict(X_test_l)
y_prob_l = model_liver.predict_proba(X_test_l)[:, 1]

print('=== LIVER DISEASE MODEL EVALUATION ===')
print(f'Accuracy  : {accuracy_score(y_test_l, y_pred_l):.4f}')
print(f'Precision : {precision_score(y_test_l, y_pred_l):.4f}')
print(f'Recall    : {recall_score(y_test_l, y_pred_l):.4f}')
print(f'F1 Score  : {f1_score(y_test_l, y_pred_l):.4f}')
print('\nClassification Report:')
print(classification_report(y_test_l, y_pred_l))

In [ ]:
# ─── Cell 19: Liver – Confusion Matrix & Feature Importance ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_l = confusion_matrix(y_test_l, y_pred_l)
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['No Disease', 'Liver Disease'],
            yticklabels=['No Disease', 'Liver Disease'])
axes[0].set_title('Liver Disease – Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

importance_df_l = pd.DataFrame({
    'Feature': X_l.columns,
    'Importance': model_liver.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(importance_df_l['Feature'], importance_df_l['Importance'],
             color=plt.cm.autumn(np.linspace(0.2, 0.9, len(importance_df_l))))
axes[1].set_title('Liver Disease – Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('../models/liver_evaluation.png', dpi=150)
plt.show()

# ROC
fpr_l, tpr_l, _ = roc_curve(y_test_l, y_prob_l)
roc_auc_l = auc(fpr_l, tpr_l)
plt.figure(figsize=(6,4))
plt.plot(fpr_l, tpr_l, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc_l:.2f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Liver Disease – ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../models/liver_roc.png', dpi=150)
plt.show()

In [ ]:
# ─── Cell 20: Save Liver Model ──────────────────────────────────────────────────
joblib.dump(model_liver, '../models/liver_model.pkl')
joblib.dump(scaler_l, '../models/liver_scaler.pkl')
joblib.dump(list(X_l.columns), '../models/liver_features.pkl')
joblib.dump({'Gender': le_gender}, '../models/liver_encoders.pkl')
print('✅ Liver disease model, scaler, encoders saved!')

---
## 📊 Model Comparison Summary

In [ ]:
# ─── Cell 21: Side-by-side ROC Comparison ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'Diabetes (AUC={auc(fpr,tpr):.2f})')
ax.plot(fpr_h, tpr_h, color='crimson', lw=2, label=f'Heart Disease (AUC={auc(fpr_h,tpr_h):.2f})')
ax.plot(fpr_l, tpr_l, color='darkorange', lw=2, label=f'Liver Disease (AUC={auc(fpr_l,tpr_l):.2f})')
ax.plot([0,1],[0,1],'k--', alpha=0.5)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison – All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../models/roc_comparison.png', dpi=150)
plt.show()

# Summary table
summary = pd.DataFrame({
    'Disease': ['Diabetes', 'Heart Disease', 'Liver Disease'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test_h, y_pred_h),
        accuracy_score(y_test_l, y_pred_l)
    ],
    'Precision': [
        precision_score(y_test, y_pred),
        precision_score(y_test_h, y_pred_h),
        precision_score(y_test_l, y_pred_l)
    ],
    'Recall': [
        recall_score(y_test, y_pred),
        recall_score(y_test_h, y_pred_h),
        recall_score(y_test_l, y_pred_l)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred),
        f1_score(y_test_h, y_pred_h),
        f1_score(y_test_l, y_pred_l)
    ],
    'ROC AUC': [
        auc(fpr, tpr),
        auc(fpr_h, tpr_h),
        auc(fpr_l, tpr_l)
    ]
}).set_index('Disease').round(4)

print('\n=== MODEL COMPARISON SUMMARY ===')
print(summary.to_string())
print('\n✅ All models trained and saved successfully!')